In [10]:
import os
import DC
import sys, threading, time
from random import random

# Connect to DisplayCluster using default host=localhost and port=1910

In [12]:
dc = DC.DC()

# Look at the content thats up 

dc.updateContent()
for i in dc.content:
    print(i, dc.content[i])

Stash the current locations of the content windows

In [21]:
orig_locations = []
for c in dc.content:
    orig_locations.append([dc.content[c][0], dc.content[c][1]])

Choose initial vectors for the content windows

In [22]:
deltas = []
for i in dc.content:
    deltas.append([(-1.0 + 2*random())/30, (-1.0 + 2*random())/30])

Heres a little code that moves each window according to their motion vectors, reversing when they hit a boundary

In [23]:
XX,YY = dc.getConfiguration()

def bump():
    global dc
    for i,c in enumerate(dc.content):
        x,y,w,h = dc.content[c]
        dx = deltas[i][0]
        dy = deltas[i][1]    
        x = x + dx;
        if ((x+w) > XX):
                x = (XX+1) - (x+w) - w;
                dx = -dx;
        elif (x < 0.0):
            x = -x;
            dx = -dx;
        y = y + dy;
        if ((y+h) > YY):
            y = (YY+1) - (y+h) - h;
            dy = -dy;
        elif (y < 0.0):
            y = -y;
            dy = -dy;
        deltas[i] = [dx, dy]
        dc.reposition(c, x, y, w, h)


Now run a thread that bumps the contents waiting for a kill flag

In [26]:
kill_thread = False

def t():
    global kill_thread
    while not kill_thread:
        bump()

idler = threading.Thread(target=t)
idler.start()


Set the kill flag and, when the thread exits, restore the contents to their original location

In [27]:
kill_thread = True
idler.join()

for i,c in enumerate(dc.content):
    dc.reposition(c, orig_locations[i][0], orig_locations[i][1], dc.content[c][2], dc.content[c][3])